In [79]:
import torch
checkpoint = torch.load("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/domain_experiment/satclip/satclip/satclip_logs/satclip/satclip/checkpoints/last-v1.ckpt", map_location="cpu")
state_dict = checkpoint["state_dict"]  # or adjust if it's under a different key

In [80]:
vit_state_dict = {
    k.replace("model.visual.", ""): v
    for k, v in state_dict.items()
    if k.startswith("model.visual.")
}


In [81]:
import sys
sys.path.append('/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/domain_experiment/satclip/satclip')
from model import VisionTransformer# or wherever it's defined


vit_model = VisionTransformer(
    input_resolution=640,
    patch_size=16,
    width=768,
    layers=12,
    heads=12,
    output_dim=768,
    in_channels=3,

)
vit_model.load_state_dict(vit_state_dict, strict=True)

<All keys matched successfully>

In [82]:
# # Set the model to evaluation mode
# vit_model.eval()
# # Example input tensor
# input_tensor = torch.randn(1, 3, 640, 640)  #
# # Adjust the size according to your model's input resolution
# # Forward pass
# with torch.no_grad():
#     output,output1 = vit_model(input_tensor)
# print("Output shape:", output.shape)  # Should be [1, 768] for the class token output
# print("Output1 shape:", output1.shape)  # Should be [1, N, 768] for the patch tokens output

In [83]:
import torch
import torch.nn as nn

class ViTBackboneForDetection(nn.Module):
    def __init__(self, vit):
        super().__init__()
        self.vit = vit
        self.out_channels = 768  # Needed by torchvision

    def forward(self, x):
        _, patch_tokens = self.vit(x)
        B, N, C = patch_tokens.shape  # [B, 1600, 768]
        print('patch_tokens shape', patch_tokens.shape)
        H = W = int(N ** 0.5)         # H = W = 40 for 640x640
        features = patch_tokens.permute(0, 2, 1).reshape(B, C, H, W)  # [B, 768, 40, 40]
        return {"0": features}


In [84]:
vit_backbone = ViTBackboneForDetection(vit_model)
# input_tensor = torch.randn(1, 3, 640, 640)  # Example input tensor
# output = vit_backbone(input_tensor)
# print("Output shape:", output["0"].shape)  # Should be [1, 768, 40, 40] for the feature map
# print("Output channels:", vit_backbone.out_channels)  # Should be 768
# print("Output feature map shape:", output["0"].shape)  # Should be [1, 768, 40, 40]
# print("Output feature map dtype:", output["0"].dtype)  # Should be float32

In [85]:
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection.transform import GeneralizedRCNNTransform
import torchvision

from torchvision.models.detection.transform import GeneralizedRCNNTransform

transform = GeneralizedRCNNTransform(
    min_size=640,
    max_size=640,
    image_mean=[0.485, 0.456, 0.406],
    image_std=[0.229, 0.224, 0.225]
)



In [86]:
print("transform", transform)

transform GeneralizedRCNNTransform(
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    Resize(min_size=(640,), max_size=640, mode='bilinear')
)


In [87]:
# Wrap it for detection
backbone = ViTBackboneForDetection(vit_model)
backbone.out_channels = 768

# Define anchor generator
anchor_generator = AnchorGenerator(
    sizes=((32, 64, 128, 256, 512),),
    aspect_ratios=((0.5, 1.0, 2.0),)
)

# Define ROI pooling
roi_pooler = torchvision.ops.MultiScaleRoIAlign(
    featmap_names=["0"],
    output_size=7,
    sampling_ratio=2
)
from torchvision.models.detection.image_list import ImageList

import torch
from torch import nn
from torchvision.models.detection.image_list import ImageList

import torch
from torch import nn
from torchvision.models.detection.image_list import ImageList

class IdentityTransform(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, images, targets=None):
        # Convert List[Tensor] to ImageList
        image_sizes = [img.shape[-2:] for img in images]
        return ImageList(torch.stack(images), image_sizes), targets

    def postprocess(self, result, image_shapes, original_image_sizes):
        # Return the results as-is; no resizing needed
        return result




In [88]:
model = FasterRCNN(
    backbone=backbone,
    num_classes=3,  
    rpn_anchor_generator=anchor_generator,
    box_roi_pool=roi_pooler,
)
model.transform=IdentityTransform()  # Use identity transform to avoid normalization issues
model

FasterRCNN(
  (transform): IdentityTransform()
  (backbone): ViTBackboneForDetection(
    (vit): VisionTransformer(
      (conv1): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
      (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (transformer): Transformer(
        (resblocks): Sequential(
          (0): ResidualAttentionBlock(
            (attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
            )
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (mlp): Sequential(
              (c_fc): Linear(in_features=768, out_features=3072, bias=True)
              (gelu): QuickGELU()
              (c_proj): Linear(in_features=3072, out_features=768, bias=True)
            )
            (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          )
          (1): ResidualAttentionBlock(
            (attn): MultiheadAttentio

In [89]:
print(model.transform)


IdentityTransform()


In [90]:
model.eval()
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
model.to(device)

# Create dummy input
input_tensor = torch.randn(3, 640, 640).to(device)

# Wrap as list and run model
with torch.no_grad():
    output = model([input_tensor])  # DO NOT wrap model() again

# Output
print("Number of detections:", len(output[0]['boxes']))
print("Boxes shape:", output[0]['boxes'].shape)
print("Labels shape:", output[0]['labels'].shape)
print("Scores shape:", output[0]['scores'].shape)


patch_tokens shape torch.Size([1, 1600, 768])
Number of detections: 100
Boxes shape: torch.Size([100, 4])
Labels shape: torch.Size([100])
Scores shape: torch.Size([100])


In [91]:
import os
import time
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign
from PIL import Image



In [92]:
import os

# Paths
image_dir = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/lucknow_airshed/images"
label_dir_yolo = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/lucknow_airshed/labels"
label_dir_voc = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/lucknow_airshed/label_aa_pascal_voc"

# # Create output directory if not exists
# os.makedirs(label_dir_voc, exist_ok=True)

# # Process each label file
# for fname in os.listdir(label_dir_yolo):
#     if not fname.endswith('.txt'):
#         continue

#     label_path = os.path.join(label_dir_yolo, fname)
#     image_path = os.path.join(image_dir, fname.replace(".txt", ".tif"))

#     if not os.path.exists(image_path):
#         continue

#     # Get image dimensions
#     from PIL import Image
#     img = Image.open(image_path)
#     w, h = img.size

#     pascal_boxes = []

#     with open(label_path, 'r') as f:
#         for line in f:
#             parts = list(map(float, line.strip().split()))
#             if len(parts) != 9:
#                 continue
#             cls = int(parts[0])
#             coords = parts[1:]
#             x_coords = coords[::2]
#             y_coords = coords[1::2]
#             x_min, x_max = min(x_coords), max(x_coords)
#             y_min, y_max = min(y_coords), max(y_coords)
#             # Clip to image size
#             x_min = max(0, min(x_min * w, w))
#             x_max = max(0, min(x_max * w, w))
#             y_min = max(0, min(y_min * h, h))
#             y_max = max(0, min(y_max * h, h))
#             pascal_boxes.append([cls, x_min, y_min, x_max, y_max])

#     # Save in Pascal VOC format: cls x_min y_min x_max y_max
#     output_path = os.path.join(label_dir_voc, fname)
#     with open(output_path, 'w') as f_out:
#         for box in pascal_boxes:
#             f_out.write(f"{int(box[0])} {box[1]:.2f} {box[2]:.2f} {box[3]:.2f} {box[4]:.2f}\n")

# label_dir_voc


In [93]:
import os
import torch
from torch.utils.data import Dataset
import numpy as np
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

# -----------------------------
# Albumentations Transform Setup
# -----------------------------
def get_train_transform(resize_crop_size=640):
    return A.Compose([
        A.Resize(height=resize_crop_size, width=resize_crop_size),
        # A.HorizontalFlip(p=0.5),
        # A.VerticalFlip(p=0.5),
        # A.RandomBrightnessContrast(p=0.5),
        # A.GaussianBlur(p=0.2),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

# ---------------------
# Dataset for TIF images
# ---------------------
class TIFRCNNDataset(Dataset):
    def __init__(self, image_dir, label_dir, transforms=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transforms = transforms
        self.image_filenames = [f for f in os.listdir(image_dir) if f.endswith('.tif')]

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_name = self.image_filenames[idx]
        image_path = os.path.join(self.image_dir, img_name)
        label_path = os.path.join(self.label_dir, os.path.splitext(img_name)[0] + ".txt")

        image = np.array(Image.open(image_path).convert("RGB"))
        boxes, labels = [], []

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = list(map(float, line.strip().split()))
                    if len(parts) == 5:
                        cls, x_min, y_min, x_max, y_max = parts
                        boxes.append([x_min, y_min, x_max, y_max])
                        labels.append(int(cls))  # background = 0

        # If no boxes, use dummy box to keep training stable (optional)
        if not boxes:
            boxes = [[0, 0, 1, 1]]
            labels = [0]

        transformed = self.transforms(
            image=image,
            bboxes=boxes,
            class_labels=labels
        )

        transformed_image = transformed['image']
        transformed_boxes = torch.tensor(transformed['bboxes'], dtype=torch.float32)
        transformed_labels = torch.tensor(transformed['class_labels'], dtype=torch.int64)

        target = {
            "boxes": transformed_boxes,
            "labels": transformed_labels
        }

        return transformed_image, target

# ---------------------
# Collate function
# ---------------------
def collate_fn(batch):
    return tuple(zip(*batch))


In [94]:


# ---------------------
# Backbone Wrapper
# ---------------------
class ViTBackboneForDetection(nn.Module):
    def __init__(self, vit):
        super().__init__()
        self.vit = vit

    def forward(self, x):
        _, tokens = self.vit(x)
        B, N, C = tokens.shape
        H = W = int(N**0.5)
        tokens = tokens.permute(0, 2, 1).reshape(B, C, H, W)
        return {"0": tokens}


In [95]:
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")

In [96]:

# ---------------------
# Main Training
# ---------------------
Dataset = TIFRCNNDataset(
    image_dir=image_dir,
    label_dir=label_dir_voc,
    transforms=get_train_transform(640)  # Resize to 640x640
)
dataloader = DataLoader(
    Dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    collate_fn=collate_fn
)


backbone = ViTBackboneForDetection(vit_model)
backbone.out_channels = 768
anchor_generator = AnchorGenerator(
    sizes=((32, 64, 128, 256, 512),),             # Only 1 feature map → 1 tuple of sizes
    aspect_ratios=((0.5, 1.0, 2.0),)              # Only 1 feature map → 1 tuple of ratios
)


roi_pooler = MultiScaleRoIAlign(
    featmap_names=["0"],
    output_size=7,
    sampling_ratio=2
)

model = FasterRCNN(
    backbone=backbone,
    num_classes=4,  # Adjust for your task
    rpn_anchor_generator=anchor_generator,
    box_roi_pool=roi_pooler)
# model_transform = GeneralizedRCNNTransform(
#     min_size=640,
#     max_size=640,
#     image_mean=[0.485, 0.456, 0.406],
#     image_std=[0.229, 0.224, 0.225]
# )
epoch_losses = []  # To store losses for each epoch
model.transform = IdentityTransform() # Set the transform for the model
# model
model.to(device)
model.train()  # Set the model to training mode
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 50
for epoch in range(num_epochs):
    start_time = time.time()
    for images, targets in dataloader:
        
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        losses.backward()
        optimizer.step()
        epoch_losses.append(losses.item())

    end_time = time.time()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {losses.item():.4f}, Time: {end_time - start_time:.2f}s")
    

    

Epoch [1/50], Loss: 0.3397, Time: 98.32s
Epoch [2/50], Loss: 0.5802, Time: 103.23s
Epoch [3/50], Loss: 0.1149, Time: 185.33s
Epoch [4/50], Loss: 0.0551, Time: 108.22s
Epoch [5/50], Loss: 0.0401, Time: 102.33s
Epoch [6/50], Loss: 0.0514, Time: 101.02s
Epoch [7/50], Loss: 0.0317, Time: 103.17s
Epoch [8/50], Loss: 0.0507, Time: 102.79s
Epoch [9/50], Loss: 0.0598, Time: 102.29s
Epoch [10/50], Loss: 0.1000, Time: 102.83s
Epoch [11/50], Loss: 0.0391, Time: 102.43s
Epoch [12/50], Loss: 0.2723, Time: 101.90s
Epoch [13/50], Loss: 11.0817, Time: 102.12s
Epoch [14/50], Loss: 158.8253, Time: 94.06s
Epoch [15/50], Loss: 319.3686, Time: 93.93s
Epoch [16/50], Loss: 0.6347, Time: 98.26s
Epoch [17/50], Loss: 0.9829, Time: 102.39s
Epoch [18/50], Loss: 0.4653, Time: 101.92s
Epoch [19/50], Loss: 16.6905, Time: 101.80s
Epoch [20/50], Loss: 13.8752, Time: 100.69s
Epoch [21/50], Loss: 23.4835, Time: 101.49s
Epoch [22/50], Loss: 9.1243, Time: 101.36s
Epoch [23/50], Loss: 150.5849, Time: 97.29s
Epoch [24/50], 

In [105]:
import os
import torch
from tqdm import tqdm
from albumentations.pytorch import ToTensorV2
import albumentations as A
from torch.utils.data import DataLoader


# Paths
image_dir = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/lucknow_airshed/images"
label_dir_voc = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/lucknow_airshed/label_aa_pascal_voc"
output_dir = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/lucknow_airshed/predictions_1"
os.makedirs(output_dir, exist_ok=True)


In [106]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image

class TIFRCNNDataset(Dataset):
    def __init__(self, image_dir, transforms=None):
        self.image_dir = image_dir
        self.transforms = transforms
        self.image_filenames = [f for f in os.listdir(image_dir) if f.endswith('.tif')]

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_name = self.image_filenames[idx]
        image_path = os.path.join(self.image_dir, img_name)
        image = Image.open(image_path).convert("RGB")
        image_np = np.array(image)

        if self.transforms:
            transformed = self.transforms(image=image_np)
            image = transformed["image"]

        return image, img_name  # return filename to write predictions later


In [107]:

# Albumentations transform
test_transform = A.Compose([
    A.Resize(640, 640),
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# Dataset
testdataset = TIFRCNNDataset(
    image_dir=image_dir,
    # label_dir=label_dir_voc,
    transforms=test_transform
)

# ---------------------
# Collate function
# ---------------------
def collate_fn(batch):
    return tuple(zip(*batch))

testdataloader = DataLoader(
    testdataset,
    batch_size=1,
    shuffle=False,
    num_workers=4,
    collate_fn=collate_fn
)



In [108]:
# model = FasterRCNN(
#     backbone=backbone,
#     num_classes=3,  # Adjust for your task
#     rpn_anchor_generator=anchor_generator,
#     box_roi_pool=roi_pooler)
# model_transform = GeneralizedRCNNTransform(
#     min_size=640,
#     max_size=640,
#     image_mean=[0.485, 0.456, 0.406],
#     image_std=[0.229, 0.224, 0.225]
# )

In [ ]:
# Inference
CONF_THRESHOLD = 0.33
model.eval()
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")

with torch.no_grad():
    for i, (images, targets) in enumerate(tqdm(testdataloader, desc="Evaluating")):
        image_tensor = images[0].to(device)
        outputs = model([image_tensor])[0]

        boxes = outputs["boxes"]
        scores = outputs["scores"]
        labels = outputs["labels"]

        keep = scores > CONF_THRESHOLD
        boxes = boxes[keep]
        scores = scores[keep]
        labels = labels[keep]

        # Handle no detections
        if len(boxes) == 0:
            continue
        # Format lines
        pred_lines = []
        for box, score, label in zip(boxes, scores, labels):
            x1, y1, x2, y2 = box.cpu().numpy()
            line = f"{label.item() - 1} {score.item():.4f} {x1:.1f} {y1:.1f} {x2:.1f} {y2:.1f}"
            pred_lines.append(line)

        # Save per-image predictions
        image_name = testdataset.image_filenames[i]
        out_file = os.path.join(output_dir, os.path.splitext(image_name)[0] + ".txt")
        with open(out_file, "w") as f:
            f.write("\n".join(pred_lines))


Evaluating: 100%|██████████| 598/598 [00:39<00:00, 14.97it/s]


: 